# Integración del csv cardekho integrado

In [2]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Carga del dataset integrado (resultado de la Sección B).
# Insumo:      data/processed/cardekho_integrado.csv
# Resultado:   DataFrame df.
# ===========================================================
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)

df = pd.read_csv("data/processed/cardekho_integrado.csv", encoding="utf-8-sig")
print(df.shape)

(15411, 42)


### D.15 Identificación los nulos

In [3]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Identificación de valores nulos por dimensión (inciso 15).
# Insumo:      df
# Resultado:   DataFrame tabla_nulos ordenado de mayor a menor.
# ===========================================================
nulos = df.isnull().sum()
porcentaje = nulos * 100 / len(df)

tabla_nulos = pd.DataFrame({"Nulos": nulos, "Porcentaje": porcentaje.round(2)})
tabla_nulos = tabla_nulos[tabla_nulos["Nulos"] > 0].sort_values("Nulos", ascending=False)
tabla_nulos

,Nulos,Porcentaje
Perdidas_Normalizadas,11433,74.19
RPM_Maximas,11423,74.12
Caballos_Fuerza,11423,74.12
MPG_Ciudad,10887,70.64
Precio,10887,70.64
Relacion_Compresion,10887,70.64
Carrera_Piston,10887,70.64
Diametro_Cilindro,10887,70.64
MPG_Carretera,10887,70.64
Tamano_Motor,10887,70.64


In [5]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Filas que sí encontraron su marca en UCI, pero sin dato de Caballos_Fuerza
# Insumo:      df
# Resultado:   Valores de nulos en Caballos_Fuerza 
# ===========================================================
df[(df["Precio"].notna()) & (df["Caballos_Fuerza"].isnull())]["Marca_Homologada"].value_counts()

Marca_Homologada
renault    536
Name: count, dtype: int64

In [6]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Filas que sí encontraron su marca en UCI, pero sin dato de Perdidas_Normalizadas
# Insumo:      df
# Resultado:   Valores de nulos en Perdidas_Normlalizadas 
# ===========================================================
df[(df["Precio"].notna()) & (df["Perdidas_Normalizadas"].isnull())]["Marca_Homologada"].value_counts()

Marca_Homologada
renault    536
isuzu       10
Name: count, dtype: int64

In [7]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Indicador de si el registro obtuvo información de UCI,
#              creado ANTES de imputar para no perder ese dato (inciso 15).
# Insumo:      df['Precio']
# Resultado:   Nueva dimensión 'Tiene_Datos_UCI' (1 = sí, 0 = no).
# ===========================================================
df["Tiene_Datos_UCI"] = df["Precio"].notna().astype(int)

df["Tiene_Datos_UCI"].value_counts()

Tiene_Datos_UCI
0    10887
1     4524
Name: count, dtype: int64

In [9]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Separar las columnas de UCI en numéricas y categóricas (inciso 15).
# Insumo:      tabla_nulos y df
# Resultado:   Listas uci_num y uci_cat con las columnas de UCI por tipo."
# ===========================================================
cols_uci = tabla_nulos.index.tolist()

uci_num = df[cols_uci].select_dtypes(include="number").columns.tolist()

uci_cat = df[cols_uci].select_dtypes(include="string").columns.tolist()

print("Numéricas  :", len(uci_num), uci_num)
print("Categóricas:", len(uci_cat), uci_cat)

Numéricas  : 18 ['Perdidas_Normalizadas', 'RPM_Maximas', 'Caballos_Fuerza', 'MPG_Ciudad', 'Precio', 'Relacion_Compresion', 'Carrera_Piston', 'Diametro_Cilindro', 'MPG_Carretera', 'Tamano_Motor', 'Num_Cilindros', 'Altura', 'Peso_Vacio', 'Ancho', 'Longitud', 'Distancia_Ejes', 'Num_Puertas', 'Nivel_Riesgo']
Categóricas: 7 ['Sistema_Combustible', 'Tipo_Motor', 'Ubicacion_Motor', 'Traccion', 'Carroceria', 'Aspiracion', 'Tipo_Combustible']


In [11]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Imputación de las dimensiones cuantitativas de UCI con la
#              mediana de cada columna (inciso 15).
# Insumo:      df, uci_num
# Resultado:   Dimensiones cuantitativas de UCI sin valores nulos.
# ===========================================================
# Mediana de cada columna
medianas = df[uci_num].median()

df[uci_num] = df[uci_num].fillna(medianas)

print("Nulos restantes en numéricas de UCI:", df[uci_num].isnull().sum().sum())
medianas.round(2)

Nulos restantes en numéricas de UCI: 0


Perdidas_Normalizadas     110.29
RPM_Maximas              5154.17
Caballos_Fuerza            81.08
MPG_Ciudad                 27.50
Precio                   9885.81
Relacion_Compresion         9.22
Carrera_Piston              3.43
Diametro_Cilindro           3.17
MPG_Carretera              32.91
Tamano_Motor              118.81
Num_Cilindros               4.08
Altura                     53.72
Peso_Vacio               2441.09
Ancho                      65.09
Longitud                  171.93
Distancia_Ejes             97.61
Num_Puertas                 3.12
Nivel_Riesgo                0.62
dtype: float64

In [12]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Sustitución de las dimensiones cualitativas de UCI con el
#              valor "Desconocido" (inciso 15).
# Insumo:      df, uci_cat
# Resultado:   Dimensiones cualitativas de UCI sin valores nulos
# ===========================================================
df[uci_cat] = df[uci_cat].fillna("Desconocido")

print("Nulos restantes en categoricas de UCI:", df[uci_cat].isnull().sum().sum())

Nulos restantes en categoricas de UCI: 0


In [13]:
# Evidencia: distribución de una columna categórica después de la sustitución
print(df["Carroceria"].value_counts(), "\n")

# Comprobación general: nulos en TODO el dataset
print("Nulos totales en el dataset:", df.isnull().sum().sum())

Carroceria
Desconocido    10887
hatchback       2814
sedan           1689
hardtop           21
Name: count, dtype: int64 

Nulos totales en el dataset: 0


### Justificación
Tratamiento de nulos (inciso 15): todos los nulos provienen de la integración. 10,887 registros (70.64%) no tienen marca en UCI, y Renault e Isuzu no tenían algunos datos en UCI. Antes de imputar se creó Tiene_Datos_UCI para identificar los valores reales. Las 18 dimensiones cuantitativas se imputaron con la mediana, por ser robusta ante valores extremos. Las 7 cualitativas se sustituyeron con "Desconocido", para no asignar categorías inventadas.

## D.16 Eliminar observaciones con nulos que impidan el análisis

In [15]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Eliminación de observaciones con nulos en dimensiones
#              esenciales de CarDekho (inciso 16).
# Insumo:      df
# Resultado:   df sin observaciones inutilizables para el análisis.
# ===========================================================
# Dimensiones esenciales: sin ellas no es posible identificar ni analizar
# un auto usado (qué auto es, de qué año, cuánto se ha usado y su precio).
cols_principales = ["Nombre", "Marca_Homologada", "Año", "Kilómetros",
                    "Combustible", "Transmisión", "Precio_Venta"]

antes = len(df)
filas_con_nulos = df[cols_principales].isnull().any(axis=1).sum()

# dropna(subset=...) elimina solo las filas con nulos en esas columnas
df = df.dropna(subset=cols_principales)

print("Observaciones antes:", antes)
print("Con nulos en dimensiones clave:", filas_con_nulos)
print("Observaciones después:", len(df))

Observaciones antes: 15411
Con nulos en dimensiones clave: 0
Observaciones después: 15411


### Justificación
Se elimina una observación solo cuando le faltan datos en dimensiones esenciales de CarDekho, porque sin ellas el auto no puede identificarse ni analizarse. Si solo le faltan datos complementarios de UCI, se conserva (ya fueron tratados en el inciso 15), porque eliminarlas implicaría perder el 70.64% del dataset principal. Ninguna observación cumplió el criterio de eliminación.

## D.17 Registros duplicados existentes

In [17]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Visualización de las columnas duplicadas (inciso 17).
# Insumo:      df
# Resultado:   df sin observaciones inutilizables para el análisis.
# ===========================================================

# 1) Duplicados considerando TODAS las columnas
print("Con todas las columnas      :", df.duplicated().sum())

# 2) Duplicados SIN Indice_Original
cols_sin_indice = [c for c in df.columns if c != "Indice_Original"]
print("Sin Indice_Original         :", df.duplicated(subset=cols_sin_indice).sum())

Con todas las columnas      : 0
Sin Indice_Original         : 167


In [18]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Identificación y eliminación de registros duplicados,
#              comparando todas las dimensiones excepto Indice_Original
#              (inciso 17).
# Insumo:      df, cols_sin_indice
# Resultado:   df sin registros duplicados.
# ===========================================================
# Evidencia: ejemplos de registros duplicados (keep=False muestra TODAS las copias)
ejemplos = df[df.duplicated(subset=cols_sin_indice, keep=False)]
print(ejemplos.sort_values(["Nombre", "Precio_Venta"])
      [["Indice_Original", "Nombre", "Año", "Kilómetros", "Precio_Venta", "Tipo_Vendedor"]].head(6))

# Eliminación: se conserva la primera aparición de cada registro
antes = len(df)
df = df.drop_duplicates(subset=cols_sin_indice, keep="first")

print("\nObservaciones antes  :", antes)
print("Duplicados eliminados:", antes - len(df))
print("Observaciones después:", len(df))

       Indice_Original   Nombre   Año  Kilómetros  Precio_Venta Tipo_Vendedor
3585              4571  Audi A4  2012       68370       1345000        Dealer
13469            17090  Audi A4  2012       68370       1345000        Dealer
10098            12907  Audi A4  2016       51000       1850000        Dealer
10492            13374  Audi A4  2016       51000       1850000        Dealer
10315            13164    BMW 5  2021        2000       4850000        Dealer
11486            14576    BMW 5  2021        2000       4850000        Dealer

Observaciones antes  : 15411
Duplicados eliminados: 167
Observaciones después: 15244


## Justificación
Se identificaron 167 registros duplicados comparando todas las dimensiones excepto Indice_Original, ya que este es un identificador técnico que hace única cada fila aunque el registro se repita. Los duplicados coinciden en modelo, año, kilómetros, precio y tipo de vendedor, por lo que corresponden al mismo vehículo publicado más de una vez. Se conservó la primera aparición y el dataset pasó de 15,411 a 15,244 observaciones.

## D.18 eliminar las dimensiones que no sea necesaria para los objetivos del análisis.

In [20]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Eliminación de dimensiones que son identificadores
#              técnicos, información redundante o no necesaria (inciso 18).
# Insumo:      df
# Resultado:   df sin las dimensiones eliminadas.
# ===========================================================
cols_eliminar = ["Indice_Original",                                  # identificador técnico
                 "Marca_Original", "Edad_Vehiculo", "Tipo_Combustible",  # redundantes
                 "Perdidas_Normalizadas", "Nivel_Riesgo", "Precio"]     # no necesarias

print("Dimensiones antes  :", df.shape[1])
df = df.drop(columns=cols_eliminar)
print("Dimensiones después:", df.shape[1])
print(df.columns.tolist())

Dimensiones antes  : 43
Dimensiones después: 36
['Nombre', 'Modelo', 'Kilómetros', 'Tipo_Vendedor', 'Combustible', 'Transmisión', 'Millaje', 'Motor', 'Potencia_Máxima', 'Asientos', 'Precio_Venta', 'Año', 'Marca', 'Marca_Homologada', 'MPG_Carretera', 'MPG_Ciudad', 'RPM_Maximas', 'Caballos_Fuerza', 'Relacion_Compresion', 'Carrera_Piston', 'Diametro_Cilindro', 'Tamano_Motor', 'Num_Cilindros', 'Peso_Vacio', 'Altura', 'Ancho', 'Longitud', 'Distancia_Ejes', 'Num_Puertas', 'Sistema_Combustible', 'Tipo_Motor', 'Ubicacion_Motor', 'Traccion', 'Carroceria', 'Aspiracion', 'Tiene_Datos_UCI']


### Justificación
Eliminación de dimensiones: se eliminaron 7 dimensiones, con lo que el dataset pasó de 43 a 36.

Identificador técnico: Indice_Original, porque solo numera las filas del archivo original y no describe al vehículo.
Información redundante: Marca_Original, porque su información ya está en Marca y Marca_Homologada; Edad_Vehiculo, porque puede obtenerse a partir de Año; y Tipo_Combustible (UCI), porque solo distingue gas/diesel y Combustible (CarDekho) contiene la misma información con mayor detalle.
Información no necesaria: Perdidas_Normalizadas y Nivel_Riesgo, porque son indicadores de seguros que no describen al vehículo ni su venta; y Precio (UCI), porque es un precio de lista promedio por marca de 1985 en dólares, no comparable con Precio_Venta (rupias, autos usados en 2021).
Identificador técnico: Indice_Original, porque solo numera las filas del archivo original y no describe al vehículo.
Información redundante: Marca_Original, porque su información ya está en Marca y Marca_Homologada; Edad_Vehiculo, porque puede obtenerse a partir de Año; y Tipo_Combustible (UCI), porque solo distingue gas/diesel y Combustible (CarDekho) contiene la misma información con mayor detalle.
Información no necesaria: Perdidas_Normalizadas y Nivel_Riesgo, porque son indicadores de seguros que no describen al vehículo ni su venta; y Precio (UCI), porque es un precio de lista promedio por marca de 1985 en dólares, no comparable con Precio_Venta (rupias, autos usados en 2021).

## D.19 Homologar categorías

In [21]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Revisión de los valores únicos de las dimensiones
#              cualitativas para detectar inconsistencias (inciso 19).
# Insumo:      df
# Resultado:   Valores únicos de cada dimensión cualitativa.
# ===========================================================
cols_texto = df.select_dtypes(exclude="number").columns.tolist()

for col in cols_texto:
    if col in ["Nombre", "Modelo"]:      # demasiados valores para listarlos
        print(f"{col}: {df[col].nunique()} valores distintos\n")
    else:
        print(f"{col}: {sorted(df[col].unique())}\n")

Nombre: 121 valores distintos

Modelo: 120 valores distintos

Tipo_Vendedor: ['Dealer', 'Individual', 'Trustmark Dealer']

Combustible: ['CNG', 'Diesel', 'Electric', 'LPG', 'Petrol']

Transmisión: ['Automatic', 'Manual']

Marca: ['Audi', 'BMW', 'Bentley', 'Datsun', 'Ferrari', 'Force', 'Ford', 'Honda', 'Hyundai', 'ISUZU', 'Isuzu', 'Jaguar', 'Jeep', 'Kia', 'Land', 'Lexus', 'MG', 'Mahindra', 'Maruti', 'Maserati', 'Mercedes-AMG', 'Mercedes-Benz', 'Mini', 'Nissan', 'Porsche', 'Renault', 'Rolls-Royce', 'Skoda', 'Tata', 'Toyota', 'Volkswagen', 'Volvo']

Marca_Homologada: ['audi', 'bentley', 'bmw', 'datsun', 'ferrari', 'force', 'ford', 'honda', 'hyundai', 'isuzu', 'jaguar', 'jeep', 'kia', 'land rover', 'lexus', 'mahindra', 'maruti', 'maserati', 'mercedes benz', 'mg', 'mini', 'nissan', 'porsche', 'renault', 'rolls royce', 'skoda', 'tata', 'toyota', 'volkswagen', 'volvo']

Sistema_Combustible: ['1bbl', '2bbl', 'Desconocido', 'idi', 'mpfi']

Tipo_Motor: ['Desconocido', 'dohc', 'ohc', 'ohcf']

Ubi

In [23]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Detección de inconsistencias de escritura en 'Nombre' y
#              'Modelo', comparando los valores distintos antes y después
#              de normalizar (minúsculas y sin espacios) (inciso 19).
# Insumo:      df['Nombre'], df['Modelo']
# Resultado:   Número de valores distintos por dimensión; si disminuye al
#              normalizar, existen categorías escritas de forma d
for col in ["Nombre", "Modelo"]:
    print(f"{col}: {df[col].nunique()} distintos -> {df[col].str.lower().str.strip().nunique()} en minúsculas y sin espacios")

Nombre: 121 distintos -> 120 en minúsculas y sin espacios
Modelo: 120 distintos -> 120 en minúsculas y sin espacios


In [24]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Homologación de categorías que representan el mismo
#              concepto pero están escritas de forma diferente (inciso 19).
# Insumo:      df['Marca'], df['Nombre']
# Resultado:   'Marca' y 'Nombre' con una representación única por concepto.
# ===========================================================
# Reglas de homologación: {valor encontrado: forma común}
reglas_marca = {
    "ISUZU": "Isuzu",                 # diferencia de mayúsculas
    "Land": "Land Rover",             # marca de dos palabras truncada
    "Mercedes-AMG": "Mercedes-Benz",  # AMG es una división de Mercedes-Benz
}

print("Marca  antes:", df["Marca"].nunique(), "| Nombre antes:", df["Nombre"].nunique())

df["Marca"] = df["Marca"].replace(reglas_marca)
# En Nombre solo se corrige la marca escrita en mayúsculas al inicio del texto
df["Nombre"] = df["Nombre"].str.replace(r"^ISUZU", "Isuzu", regex=True)

print("Marca después:", df["Marca"].nunique(), "| Nombre después:", df["Nombre"].nunique())
print(sorted(df["Marca"].unique()))

Marca  antes: 32 | Nombre antes: 121
Marca después: 30 | Nombre después: 120
['Audi', 'BMW', 'Bentley', 'Datsun', 'Ferrari', 'Force', 'Ford', 'Honda', 'Hyundai', 'Isuzu', 'Jaguar', 'Jeep', 'Kia', 'Land Rover', 'Lexus', 'MG', 'Mahindra', 'Maruti', 'Maserati', 'Mercedes-Benz', 'Mini', 'Nissan', 'Porsche', 'Renault', 'Rolls-Royce', 'Skoda', 'Tata', 'Toyota', 'Volkswagen', 'Volvo']


## D.20 Verificación final de la limpieza

In [25]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Verificación final de la limpieza: nulos, duplicados,
#              tipos de dato y valores inválidos (inciso 20).
# Insumo:      df
# Resultado:   Reporte de comprobaciones de la Sección D.
# ===========================================================
# 1) Nulos: no debe quedar ninguno
print("1) Nulos totales           :", df.isnull().sum().sum())

# 2) Duplicados: se revisa de nuevo, por si la homologación del inciso 19
#    volvió idénticos registros que antes se escribían distinto
print("2) Registros duplicados    :", df.duplicated().sum())

# 3) Tipos de dato: cuántas columnas hay de cada tipo
print("3) Tipos de dato:\n", df.dtypes.value_counts(), "\n")

# 4) Valores inválidos en dimensiones clave
print("4) Precio_Venta <= 0       :", (df["Precio_Venta"] <= 0).sum())
print("   Kilómetros < 0          :", (df["Kilómetros"] < 0).sum())
print("   Año fuera de 1990-2021  :", (~df["Año"].between(1990, 2021)).sum())
print("   'Desconocido' sin dato UCI coincide:",
      ((df["Carroceria"] == "Desconocido") == (df["Tiene_Datos_UCI"] == 0)).all())

print("\nDimensiones finales:", df.shape)

1) Nulos totales           : 0
2) Registros duplicados    : 0
3) Tipos de dato:
 float64    17
str        13
int64       6
Name: count, dtype: int64 

4) Precio_Venta <= 0       : 0
   Kilómetros < 0          : 0
   Año fuera de 1990-2021  : 0
   'Desconocido' sin dato UCI coincide: True

Dimensiones finales: (15244, 36)


In [26]:
# ===========================================================
# Autor:       Magaly
# Fecha:       05 octubre 2026
# Descripción: Exportación del dataset limpio para la Sección E.
# Insumo:      df
# Resultado:   data/processed/cardekho_limpio.csv
# ===========================================================
df.to_csv("data/processed/cardekho_limpio.csv", index=False, encoding="utf-8-sig")
print("Exportado:", df.shape)

Exportado: (15244, 36)
